In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

from arch import arch_model

# Problem 1

## (a) Unit root test

In [ ]:
df = pd.read_csv('data/us_gdp.csv', parse_dates=['observation_date'], index_col='observation_date')
df = df.asfreq('QS')  # Quarterly Start frequency
print(df.head(3))
print(df.tail(3))
print(f"\ndescription:\n{df['GDP'].describe()}")
print(f"\nna counts:\n{df.isna().sum()}")

In [ ]:
gdp_series = df['GDP'] 
result = adfuller(gdp_series)

print(f'ADF Statistic: {result[0]:.4f}')
print(f'p-value: {result[1]:.4f}')
print('Critical Values:')
for key, value in result[4].items():
    print(f'   {key}: {value:.4f}')

# alpha = 0.05
if result[1] < 0.05:
    print("Reject H0 : Stationary")
else:
    print("Non-Reject H0 : Non-stationary, unit root exists")

In [ ]:
# differencing
gdp_diff1 = gdp_series.diff().dropna()
result_diff1 = adfuller(gdp_diff1)
print(f'\nADF Statistic (1st difference): {result_diff1[0]:.4f}')
print(f'p-value (1st difference): {result_diff1[1]:.4f}')
print('Critical Values (1st difference):')
for key, value in result_diff1[4].items():
    print(f'   {key}: {value:.4f}')
if result_diff1[1] < 0.05:
    print("Reject H0 : Stationary after 1st differencing")
else:
    print("Non-Reject H0 : Still non-stationary after 1st differencing")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
gdp_series.plot(ax=axes[0], title='US GDP Over Time')
gdp_diff1.plot(ax=axes[1], color='orange', title='First Difference of US GDP')
axes[1].axhline(0, color='orange', linestyle='--')
plt.tight_layout()
plt.show()

## (b) Decide (p, i, q)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_acf(gdp_diff1, ax=axes[0], lags=20)
plot_pacf(gdp_diff1, ax=axes[1], lags=20)
plt.show()

In [ ]:
results = []
for p, q in product(range(0, 4), range(0, 4)):
    try:
        model = ARIMA(gdp_series, order=(p, 1, q))
        model_fit = model.fit()
        results.append({'order': (p,1,q), 'AIC': model_fit.aic, 'BIC': model_fit.bic})
    except:
        continue
results_df = pd.DataFrame(results).sort_values('AIC')
print(results_df.head())

## (c) Coefficient Estimation

In [ ]:
best_model = ARIMA(gdp_series, order=(2, 1, 1)).fit()
print(best_model.summary())

In [ ]:
print(best_model.params)

## (d) Residual White Noise Check (Ljung-Box Test)

In [ ]:
residuals = best_model.resid

lb_test = acorr_ljungbox(residuals, lags=[10], return_df=True)
print("Ljung-Box Test (lag=10):")
print(lb_test)
print()
if lb_test['lb_pvalue'].values[0] > 0.05:
    print("p-value > 0.05 --> Fail to reject H0: residuals are white noise --> model is adequate")
else:
    print("p-value <= 0.05 --> Reject H0: residuals are NOT white noise --> model may be inadequate")

print("\nTesting multiple lags:")
lags = [5, 10, 15, 20]
for lag in lags:
    lb_test = acorr_ljungbox(residuals, lags=[lag], return_df=True)
    print(f"Ljung-Box Test (lag={lag}): stat={lb_test['lb_stat'].values[0]:.4f}, p-value={lb_test['lb_pvalue'].values[0]:.4f}")
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(residuals)
axes[0].set_title('Residuals')
axes[0].axhline(0, color='red', linestyle='--')
plot_acf(residuals.values, ax=axes[1], lags=20)
axes[1].set_title('ACF of Residuals')
plt.tight_layout()
plt.show()

## (e) k-step Ahead Forecasting (k = 1 to 5)

In [ ]:
forecast_result = best_model.get_forecast(steps=5)
forecast_mean = forecast_result.predicted_mean
forecast_ci = forecast_result.conf_int(alpha=0.05)

forecast_df = pd.DataFrame({
    'k': range(1, 6),
    'Date': forecast_mean.index,
    'Forecast': forecast_mean.values,
    'Lower 95% CI': forecast_ci.iloc[:, 0].values,
    'Upper 95% CI': forecast_ci.iloc[:, 1].values
})
print(forecast_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(gdp_series[-30:], label='Observed', color='blue')
ax.plot(forecast_mean, 'ro-', label='Forecast')
ax.fill_between(forecast_ci.index,
                forecast_ci.iloc[:, 0],
                forecast_ci.iloc[:, 1],
                alpha=0.3, color='red', label='95% CI')
ax.legend()
ax.set_title('ARIMA(2,1,1) 5-step Ahead Forecast')
plt.show()

# Problem 2

In [ ]:
# log return
df2 = pd.read_csv('data/three_month_returns.txt', sep='\s+', parse_dates=['date'], index_col='date')
df2.index = pd.DatetimeIndex(df2.index).to_period('M')
# Yt = log(1 + rtn)
df2['log_rtn'] = np.log(1 + df2['rtn'])
log_rtn = df2['log_rtn']

print(df2.head())
print(f"\nshape: {df2.shape}")
print(f"\nlog_rtn describe:\n{log_rtn.describe()}")

In [ ]:
# Fit mean model: ARIMA on log return (select order by AIC)
results2 = []
for p, q in product(range(0, 4), range(0, 4)):
    try:
        m = ARIMA(log_rtn, order=(p, 0, q)).fit()
        results2.append({'order': (p, 0, q), 'AIC': m.aic, 'BIC': m.bic})
    except:
        continue

results2_df = pd.DataFrame(results2).sort_values('AIC')
print(results2_df.head(5))

## (a) ARCH-LM Test

In [ ]:
best_order_2 = results2_df.iloc[0]['order']
print(f"Best mean model: ARIMA{best_order_2}")

mean_model = ARIMA(log_rtn, order=best_order_2).fit()
resid2 = mean_model.resid

lm_stat, lm_pvalue, f_stat, f_pvalue = het_arch(resid2, nlags=12)
print(f"\nARCH-LM Test (lags=12):")
print(f"LM Statistic : {lm_stat:.4f}")
print(f"LM p-value   : {lm_pvalue:.4f}")
print(f"F statistic  : {f_stat:.4f}")
print(f"F p-value    : {f_pvalue:.4f}")
if lm_pvalue < 0.05:
    print("p-value < 0.05 --> Reject H0: ARCH effect exists --> GARCH model is needed")
else:
    print("p-value >= 0.05 --> Fail to reject H0: No ARCH effect")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(resid2.values ** 2)
axes[0].set_title('Squared Residuals')
plot_acf(resid2.values ** 2, ax=axes[1], lags=20)
axes[1].set_title('ACF of Squared Residuals')
plt.tight_layout()
plt.show()

## (b) GARCH(1,1) Model

In [ ]:
# Scale by 100
garch = arch_model(resid2 * 100, mean='Zero', vol='GARCH', p=1, q=1, dist='normal')
garch_fit = garch.fit(disp='off')
print(garch_fit.summary())

# Report
print("\n" + "="*60)
print("Fitted GARCH(1,1) System:")
print("Mean equation  : Yt = ARIMA(2,0,2) residual = εt")
params = garch_fit.params
print(f"Variance equation: σ²t = {params['omega']:.6f} + {params['alpha[1]']:.4f}·ε²t-1 + {params['beta[1]']:.4f}·σ²t-1")
print("="*60)

## (c) GARCH Residual Check

In [ ]:
# Standardized residuals: zt = εt / σ̂t
std_resid = garch_fit.resid / garch_fit.conditional_volatility
lags = [5, 10, 15, 20]
# (1) Ljung-Box on zt: check if mean structure is fully captured
print("\nTesting multiple lags for zt:")
print("Ljung-Box Test on zt:")
for lag in lags:
    lb_z = acorr_ljungbox(std_resid, lags=[lag], return_df=True)
    print(f"Ljung-Box on zt (lag={lag}): stat={lb_z['lb_stat'].values[0]:.4f}, p-value={lb_z['lb_pvalue'].values[0]:.4f}")

# (2) Ljung-Box on zt^2: check if ARCH effect is removed
print("\nLjung-Box Test on zt^2:")
for lag in lags:
    lb_z2 = acorr_ljungbox(std_resid**2, lags=[lag], return_df=True)
    print(f"Ljung-Box on zt^2 (lag={lag}): stat={lb_z2['lb_stat'].values[0]:.4f}, p-value={lb_z2['lb_pvalue'].values[0]:.4f}")

# (3) ARCH-LM test on standardized residuals
lm2, lmp2, _, _ = het_arch(std_resid, nlags=12)
print(f"\nARCH-LM on standardized residuals: LM={lm2:.4f}, p={lmp2:.4f}")
if lmp2 > 0.05:
    print("p > 0.05 --> No remaining ARCH effect --> GARCH(1,1) is adequate")
else:
    print("p < 0.05 --> Still has ARCH effect --> consider higher order GARCH")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(std_resid.values, ax=axes[0], lags=20)
axes[0].set_title('ACF of Standardized Residuals (z_t)')
plot_acf(std_resid.values**2, ax=axes[1], lags=20)
axes[1].set_title('ACF of z_t^2')
plt.tight_layout()
plt.show()

## (d) 1-step to 5-step Ahead Volatility Forecasts (origin: 2008/12/31)

In [ ]:
# Forecast variance for k=1 to 5 from last observation (2008/12/31)
vol_forecast = garch_fit.forecast(horizon=5, reindex=False)
var_forecasts = vol_forecast.variance.iloc[-1]   # σ²  (scaled by 100²)
std_forecasts = np.sqrt(var_forecasts) / 100     # σ   (back to original scale)

vol_df = pd.DataFrame({
    'k': range(1, 6),
    'Variance σ² (scaled)': var_forecasts.values,
    'Volatility σ (original)': std_forecasts.values
})
print("Volatility Forecasts from 2008/12/31:")
print(vol_df.to_string(index=False))

# historical conditional volatility + 5-step forecast
hist_vol = garch_fit.conditional_volatility / 100 
fc_idx = pd.period_range(log_rtn.index[-1] + 1, periods=5, freq='M').to_timestamp()
hist_idx = log_rtn.index.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(hist_idx[-120:], hist_vol.values[-120:], color='steelblue',
        label='Historical conditional σ_t')
ax.plot(fc_idx, std_forecasts.values, 'ro-', label='Forecast σ_{T+k}')
ax.axvline(hist_idx[-1], color='gray', linestyle='--', linewidth=0.8,
           label='Forecast origin (2008-12)')
ax.set_title('GARCH(1,1) Conditional Volatility — last 10 years + 5-step forecast')
ax.set_ylabel('σ (original scale)')
ax.legend()
plt.tight_layout()
plt.show()

## (e) 1-step to 5-step Ahead Yt Forecasts (origin: 2008/12/31)

In [ ]:
# Yt point forecast comes from ARIMA mean model
# CI uses GARCH volatility: ±1.96 * σ_k
arima_fc = mean_model.get_forecast(steps=5)
yt_mean = arima_fc.predicted_mean.values

yt_df = pd.DataFrame({
    'k': range(1, 6),
    'Yt Forecast': yt_mean,
    'Lower 95% CI': yt_mean - 1.96 * std_forecasts.values,
    'Upper 95% CI': yt_mean + 1.96 * std_forecasts.values
})
print("Yt Forecasts from 2008/12/31:")
print(yt_df.to_string(index=False))

# historical Yt + 5-step forecast with GARCH-based CI 
hist_idx = log_rtn.index.to_timestamp()
fc_idx = pd.period_range(log_rtn.index[-1] + 1, periods=5, freq='M').to_timestamp()

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(hist_idx[-60:], log_rtn.values[-60:], color='steelblue',
        label='Historical Y_t (log-return)')
ax.plot(fc_idx, yt_mean, 'ro-', label='Point forecast')
ax.fill_between(fc_idx,
                yt_mean - 1.96 * std_forecasts.values,
                yt_mean + 1.96 * std_forecasts.values,
                alpha=0.3, color='red', label='95% CI (GARCH)')
ax.axhline(0, color='gray', linewidth=0.6)
ax.axvline(hist_idx[-1], color='gray', linestyle='--', linewidth=0.8,
           label='Forecast origin (2008-12)')
ax.set_title('ARIMA(2,0,2)+GARCH(1,1): 5-step Y_t Forecast with Dynamic CI')
ax.set_ylabel('Y_t (log-return)')
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

# Problem 3 - Seasonal ARIMA

In [ ]:
df3 = pd.read_csv('data/housing_sales.csv', index_col=0)
df3.index = pd.date_range(start='1973-01', periods=len(df3), freq='MS')
hsales = df3['x']

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
axes[0].plot(hsales)
axes[0].set_title('Monthly Housing Sales (Original)')
axes[1].plot(np.log(hsales))
axes[1].set_title('Monthly Housing Sales (Log)')
plt.tight_layout()
plt.show()

print(f"Observations: {len(hsales)}")
print(f"Period: {hsales.index[0].date()} to {hsales.index[-1].date()}")

## (a) Transformation Check

In [ ]:
yearly_mean = hsales.resample('Y').mean()
yearly_std  = hsales.resample('Y').std()

log_sales = np.log(hsales)
ly_mean = log_sales.resample('Y').mean()
ly_std  = log_sales.resample('Y').std()

corr_orig = np.corrcoef(yearly_mean, yearly_std)[0,1]
corr_log  = np.corrcoef(ly_mean, ly_std)[0,1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(yearly_mean, yearly_std)
axes[0].set_xlabel('Yearly Mean'); axes[0].set_ylabel('Yearly Std')
axes[0].set_title('Original: Mean vs Std (correlation={:.4f})'.format(corr_orig))

axes[1].scatter(ly_mean, ly_std)
axes[1].set_xlabel('Yearly Mean'); axes[1].set_ylabel('Yearly Std')
axes[1].set_title('Log: Mean vs Std (correlation={:.4f})'.format(corr_log))
plt.tight_layout()
plt.show()

print(f"Correlation (Mean vs Std) - Original: {corr_orig:.4f}")
print(f"Correlation (Mean vs Std) - Log     : {corr_log:.4f}")

## (b) Stationarity & Differencing

In [ ]:
y = log_sales  # log-transformed series

def adf_report(series, label):
    r = adfuller(series.dropna())
    print(f"[{label}]  ADF={r[0]:.4f}  p={r[1]:.4f}  --> {'Stationary' if r[1]<0.05 else 'Non-stationary'}")

adf_report(y, 'log(hsales)')

# Seasonal differencing: remove yearly pattern (lag=12)
y_sdiff = y.diff(12).dropna()
adf_report(y_sdiff, 'Seasonal diff (D=1)')

# Regular differencing on top
y_sdiff_diff = y_sdiff.diff(1).dropna()
adf_report(y_sdiff_diff, 'Seasonal + Regular diff (D=1, d=1)')

# Plot all three for visual check
fig, axes = plt.subplots(3, 1, figsize=(14, 9))
y.plot(ax=axes[0], title='log(hsales)')
y_sdiff.plot(ax=axes[1], title='Seasonal diff (lag=12): d=0, D=1')
y_sdiff_diff.plot(ax=axes[2], title='Seasonal + Regular diff: d=1, D=1')
plt.tight_layout()
plt.show()

# Seasonal decomposition
from statsmodels.tsa.seasonal import seasonal_decompose
decomp = seasonal_decompose(y, model='additive', period=12)
fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
decomp.observed.plot(ax=axes[0], title='Observed');  axes[0].set_ylabel('Observed')
decomp.trend.plot(ax=axes[1], title='Trend');         axes[1].set_ylabel('Trend')
decomp.seasonal.plot(ax=axes[2], title='Seasonal');   axes[2].set_ylabel('Seasonal')
decomp.resid.plot(ax=axes[3], title='Residual');      axes[3].set_ylabel('Residual')
plt.tight_layout()
plt.show()

## (c) Identify SARIMA Order via ACF/PACF + AIC

In [ ]:
# ACF/PACF on the stationary series to identify p,q,P,Q
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y_sdiff_diff, ax=axes[0], lags=40)
axes[0].set_title('ACF (d=1, D=1)')
plot_pacf(y_sdiff_diff, ax=axes[1], lags=40)
axes[1].set_title('PACF (d=1, D=1)')
plt.tight_layout()
plt.show()

# AIC grid search over candidate SARIMA orders
# burn-in = d + D*s = 1 + 1*12 = 13 
burnin = 13
sarima_results = []
for p, q, P, Q in product(range(0, 3), range(0, 3), range(0, 3), range(0, 3)):
    try:
        m = SARIMAX(y, order=(p, 1, q), seasonal_order=(P, 1, Q, 12)).fit(disp=False)
        resid_trim = m.resid.iloc[burnin:]
        lb_p = acorr_ljungbox(resid_trim, lags=[12], model_df=0, return_df=True)['lb_pvalue'].values[0]
        pvals = m.pvalues.drop('sigma2', errors='ignore')
        max_param_p = pvals.max() if len(pvals) > 0 else np.nan
        sarima_results.append({'order': (p, 1, q),
                               'seasonal': (P, 1, Q, 12),
                               'AIC': m.aic, 'BIC': m.bic,
                               'LB12 p': lb_p,
                               'Max param p': max_param_p})
    except Exception:
        continue

sarima_df = pd.DataFrame(sarima_results).sort_values('AIC').reset_index(drop=True)
print(f"Total candidates evaluated: {len(sarima_df)}")
print(sarima_df.head(10).to_string())


## (d) Estimate Parameters & Residual Check

In [ ]:
sarima_df = sarima_df[sarima_df['LB12 p'] > 0.05]  
sarima_df = sarima_df[sarima_df['Max param p'] < 0.05]
print(f"\nCandidates after filtering (LB12 p > 0.05 and all params p < 0.05): {len(sarima_df)}")
print(sarima_df.head(10))

In [ ]:
best = sarima_df.iloc[0]
best_order = best['order']
best_seasonal = best['seasonal']
print(f"Best model: SARIMA{best_order}x{best_seasonal}")

sarima_fit = SARIMAX(y, order=best_order, seasonal_order=best_seasonal).fit(disp=False)
print(sarima_fit.summary())

# Residual diagnostics
resid_s = sarima_fit.resid
print("\nLjung-Box Test:")
for lag in [6, 12, 18, 24]:
    resid_trim = sarima_fit.resid.iloc[burnin:]
    lb_s = acorr_ljungbox(resid_trim, lags=[lag], model_df=0, return_df=True)
    print(f"Ljung-Box on residuals (lag={lag}): stats={lb_s['lb_stat'].values[0]:.4f} p-value={lb_s['lb_pvalue'].values[0]:.4f}")

import scipy.stats as stats
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(resid_s.values, color='steelblue')
axes[0, 0].axhline(0, color='red', linestyle='--', linewidth=0.8)
axes[0, 0].set_title('Residuals')
plot_acf(resid_s.values, ax=axes[0, 1], lags=40)
axes[0, 1].set_title('ACF of Residuals')
axes[1, 0].hist(resid_s.values, bins=30, color='steelblue', edgecolor='white')
axes[1, 0].set_title('Residual Histogram')
stats.probplot(resid_s.values, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('QQ Plot of Residuals')
plt.tight_layout()
plt.show()

## (e) Forecast Next 24 Months

In [ ]:
fc = sarima_fit.get_forecast(steps=24)
fc_mean_log = fc.predicted_mean
fc_ci_log   = fc.conf_int(alpha=0.05)

# Back-transform from log scale
fc_mean = np.exp(fc_mean_log)
fc_lower = np.exp(fc_ci_log.iloc[:, 0])
fc_upper = np.exp(fc_ci_log.iloc[:, 1])

fc_df = pd.DataFrame({
    'Date': fc_mean.index,
    'Forecast': fc_mean.values,
    'Lower 95% CI': fc_lower.values,
    'Upper 95% CI': fc_upper.values
})
print(fc_df.to_string(index=False))

# full historical series + 24-month forecast
fig, ax = plt.subplots(figsize=(14, 5))
hsales.plot(ax=ax, label='Historical Sales', color='black', linewidth=0.9)
fc_mean.plot(ax=ax, label='Forecast', color='red', marker='o', markersize=3)
ax.fill_between(fc_ci_log.index, fc_lower, fc_upper,
                alpha=0.25, color='red', label='95% CI')
ax.axvline(hsales.index[-1], color='gray', linestyle='--', linewidth=0.8)
ax.set_title(f'SARIMA{best_order}x{best_seasonal} — 24-Month Ahead Forecast (Full History)')
ax.set_ylabel('Sales (thousands)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

# last 5 years + forecast 
fig, ax = plt.subplots(figsize=(14, 5))
hsales[-60:].plot(ax=ax, label='Observed (last 5y)', color='black')
fc_mean.plot(ax=ax, label='Forecast', color='red', marker='o', markersize=4)
ax.fill_between(fc_ci_log.index, fc_lower, fc_upper,
                alpha=0.25, color='red', label='95% CI')
ax.axvline(hsales.index[-1], color='gray', linestyle='--', linewidth=0.8)
ax.set_title(f'SARIMA{best_order}x{best_seasonal} — 24-Month Forecast (Zoomed)')
ax.set_ylabel('Sales (thousands)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()